[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/cayley_klein/cayley_klein.ipynb)

# Cayley–Klein Geometry

This example constructs hyperbolic distances, corner angles, perpendicular reflections, and concentric spheres inside the Beltrami–Klein disc from one chosen quadric. Projective geometry defines point joins and plane meets without distances or angles; selecting one quadric, called the absolute, supplies the entire metric ([Gunn, 2011](https://depositonce.tu-berlin.de/bitstreams/5a02e6fc-d9c3-4e6f-a5ab-f4a970e81209/download)). With the unit circle as the absolute, straight chords inside the disc act as hyperbolic geodesics whose lengths diverge toward the boundary and whose triangle angles sum to less than a half turn, while flipping the weight sign yields elliptic geometry from the same constructions.

The absolute is modelled as a polarity map `Plane <- Point` built as a signed dyad sum of the coordinate planes, `(coordinate_planes * (coordinate_planes & Point) * absolute_signs).sum(axis=0)`, and `absolute.solve(Plane)` maps planes back to their poles. Normalized pairings `-(absolute(Point) & Point)` and `Plane & pole_map(Plane)` yield hyperbolic distances and corner angles, joining a point to `absolute.solve(side)` drops the perpendicular through the pole, harmonic homology `point - pole * (2 * (side & point) / (side & pole))` reflects across the plane, and clearing the square root turns each sphere of fixed radius into a polarity map `polar * (polar & Point) - (polar & centre) * cosh(radii)**2 * absolute`.

In homogeneous projective matrix notation, the absolute is written as a symmetric $3 \times 3$ matrix $C$ with inverse adjugate matrix $C^{-1}$ on dual row vectors, Cayley's distance and plane angle separate into quadratic forms $\cosh d = -\mathbf p^{\mathsf T} C \mathbf q / \sqrt{(\mathbf p^{\mathsf T} C \mathbf p)(\mathbf q^{\mathsf T} C \mathbf q)}$ and $\cos\theta = \mathbf l^{\mathsf T} C^{-1} \mathbf m / \sqrt{(\mathbf l^{\mathsf T} C^{-1} \mathbf l)(\mathbf m^{\mathsf T} C^{-1} \mathbf m)}$, joins and meets require $3 \times 3$ cross-product skew matrices $[\mathbf p]_\times \mathbf q$ and $[\mathbf l]_\times \mathbf n$, and the sphere locus becomes the homogeneous matrix pencil $(C\mathbf c)(C\mathbf c)^{\mathsf T} - (\mathbf c^{\mathsf T} C \mathbf c)\cosh^2 r\,C$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from examples.quadrics.cayley_klein import render
from examples.quadrics.cayley_klein.scenarios import (
    absolute_signs,
    centre_coordinates,
    probe_coordinates,
    sphere_radii,
    triangle_simplices,
    vertex_coordinates,
)
from examples.util.animation import save_figure
from numga import Extensor, NumpyContext, stack
from numga.algebras import PGA2D as ga
from numga.topology import Topology

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
# Points are intrinsic antivectors and planes are extrinsic vectors with a side, as polars and mirrors
# carry; joining two points gives an intrinsic line, which becomes a sided plane when multiplied by the
# triangle's intrinsic winding scalar.
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Line = ga.gatype.antibivector()                     # Antibivector-
Polarity = ga.gatype((Plane, Point))                # Plane+ <- Point-

## Absolute polarity

Each coordinate plane paired with a point defines a rank-one map from points to planes, and summing the three coordinate planes with signs `(1, 1, -1)` forms the polarity of the unit circle.

In [ ]:
coordinate_planes = stack([mv.x, mv.y, mv.w])                                      # [3] Plane+
absolute = (coordinate_planes * (coordinate_planes & Point) * absolute_signs).sum(axis=0)  # [] Plane+ <- Point-
print("absolute polarity (Plane+ <- Point-):\n", absolute.coefficient_table())

## Hyperbolic triangle and angle defect

Multiplying the joins of the triangle's vertices by the corner winding turns intrinsic lines into inward-oriented planes that the inverse polarity can pair. Normalized pairings through the absolute and its inverse yield the opposite side lengths and corner angles, and the Gauss–Bonnet angle defect gives the hyperbolic area.

In [ ]:
topology = Topology(len(vertex_coordinates), triangle_simplices)
vertices = mv.point(vertex_coordinates).field()                                    # Point-[V]

In [ ]:
def normalized_pairing(pairing: Extensor, first: Extensor, second: Extensor) -> Scalar:
    """Normalized bilinear pairing of `first` and `second` over the root of their self-pairings (`[...] Scalar+`)."""
    return pairing(first, second) / (pairing(first, first) * pairing(second, second)).square_root()


def triangle_metrics(
    topology: Topology, absolute: Polarity, vertices: Point,
) -> tuple[Plane, Scalar, Scalar, Scalar]:
    """Inward side planes (`[3] Plane+[F]`), corner angles (`[3] Scalar+[F]`), opposite side lengths
    (`[3] Scalar+[F]`), and hyperbolic area (`[] Scalar+[F]`) of triangles against `absolute`."""
    pole_map = absolute.solve(Plane)                                       # [] Point- <- Plane+

    # Cayley's hyperbolic distance is arccosh of the negative normalized point pairing, and the corner
    # angle is arccos of the normalized plane pairing through the inverse polarity.
    distance_pairing = -(absolute(Point) & Point)                          # [] Scalar+ <- (Point-, Point-)
    angle_pairing = Plane & pole_map(Plane)                                # [] Scalar+ <- (Plane+, Plane+)

    corners = topology.incidence[2, 0] * vertices                          # [3] Point-[F]
    facing = topology.off[2, 0] * vertices                                 # [2, 3] Point-[F]
    # Multiplying each point join by the corner's intrinsic winding turns the intrinsic line into an
    # extrinsic plane oriented toward the interior; the common winding cancels in every angle ratio.
    corner_windings = corners & facing[0] & facing[1]                      # [] Scalar-[F]
    sides = (facing[0] & facing[1]) * corner_windings                      # [3] Plane+[F]
    first_sides = (corners & facing[0]) * corner_windings                  # [3] Plane+[F]
    second_sides = (corners & facing[1]) * corner_windings                 # [3] Plane+[F]
    angles = normalized_pairing(angle_pairing, first_sides, second_sides).clip(-1.0, 1.0).arccos()  # [3] Scalar+[F]
    lengths = normalized_pairing(distance_pairing, facing[0], facing[1]).clip(1.0, np.inf).arccosh()  # [3] Scalar+[F]
    area = np.pi - angles.sum(axis=0)                                      # [] Scalar+[F]
    return sides, angles, lengths, area


sides, angles, lengths, area = triangle_metrics(topology, absolute, vertices)

In [ ]:
print("angles:", angles.to_array().ravel(), "sum:", angles.sum(axis=0).to_array().ravel())
print("side lengths:", lengths.to_array().ravel(), "area:", area.to_array().ravel())

## Perpendicular drop and harmonic reflection

Every line perpendicular to a plane passes through the plane's pole outside the absolute, so joining a probe point to that pole drops the perpendicular and meeting it with the plane locates the foot. Harmonic homology centred on the pole reflects the point across the plane while preserving its hyperbolic distance to the foot.

In [ ]:
probe = mv.point(probe_coordinates)                                                # [] Point-

In [ ]:
def perpendicular_reflection(
    absolute: Polarity, side: Plane, point: Point,
) -> tuple[Point, Line, Point, Point]:
    """Pole of `side`, perpendicular line through `point`, its foot on `side`, and the harmonic
    reflection of `point` across `side` (`[] Point-`, `[] Line-`, `[] Point-`, `[] Point-`)."""
    # Every line perpendicular to a plane passes through its pole, so joining the point to the pole
    # drops the perpendicular, and harmonic homology centred at the pole reflects across the plane.
    pole = absolute.solve(side)                                            # [] Point-
    normal = point & pole                                                  # [] Line-
    foot = side ^ normal                                                   # [] Point-
    homology_step = 2 * (side & point) / (side & pole)                     # [] Scalar+
    reflected_point = point - pole * homology_step                         # [] Point-
    return pole, normal, foot, reflected_point


pole, normal, foot, reflected_point = perpendicular_reflection(absolute, sides[0], probe)

## Concentric spheres

Fixing the normalized pairing with a centre and clearing the square root turns each hyperbolic sphere into a polarity map whose null locus is drawn directly by the quadric renderer.

In [ ]:
n_centres = len(centre_coordinates)
n_radii = len(sphere_radii)
centres = mv.point(centre_coordinates)                                             # [n_centres] Point-

In [ ]:
def concentric_spheres(absolute: Polarity, centres: Point, radii: np.ndarray) -> Polarity:
    """Concentric sphere quadrics (`[..., n_radii] Plane+ <- Point-`) of `radii` (`[n_radii]`) around `centres`."""
    # Clearing the square root in the fixed-distance invariant turns each sphere into a polarity map
    # whose null locus is P & sphere(P) == 0.
    n_radii = len(radii)
    centre = centres[..., None]                                            # [..., 1] Point-
    polar = absolute(centre)                                               # [..., 1] Plane+
    return polar * (polar & Point) - (polar & centre) * np.cosh(radii) ** 2 * absolute  # [..., n_radii] Plane+ <- Point-


sphere_quadrics = concentric_spheres(absolute, centres, sphere_radii)              # [n_centres, n_radii] Plane+ <- Point-
save_figure(
    render.hyperbolic_figure(
        absolute, vertices, sides, angles, probe, foot, normal, reflected_point, pole, sphere_quadrics, centres,
    ),
    "cayley_klein",
);

On the left, the blue triangle lies inside the black unit-circle absolute with corner angles summing below 180 degrees, while the red perpendicular joins the red probe point through its dark-red square foot and orange reflection to the hypotenuse's purple-diamond pole outside the disc. On the right, equal steps of hyperbolic radius around the two dark centres appear as blue and orange ellipses crowding toward the boundary.

In [ ]:
# checks
# Hyperbolic triangles have positive angle defect, the pole of the perpendicular lies on the side,
# and harmonic reflection across the side preserves hyperbolic distance to the foot.
distance_pairing = -(absolute(Point) & Point)                                      # [] Scalar+ <- (Point-, Point-)
assert (area > 0).all()
assert ((sides[0] & (absolute(probe) ^ absolute(pole))).abs() < 1e-12).all()
distance_difference = (
    normalized_pairing(distance_pairing, probe, foot)
    - normalized_pairing(distance_pairing, reflected_point, foot)
)
assert (distance_difference.abs() < 1e-12).all()